# SARIMA

Вход — `df` из двух столбцов: дата, затем число. Имена столбцов любые, на дату нужна одна строка. Для остатков возьми столбцы `REP_DT` и `OST_VAL` по одной валюте. Создай `df` в этом же ядре.

Прогноз строится по дням до той же даты следующего месяца включительно. `s = 7` задаёт недельную сезонность. Пропущенные даты остаются пропусками на дневной сетке. Если нужно установить пакеты, выполни `%pip install pandas statsmodels matplotlib` в отдельной ячейке.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

ORDER = (1, 1, 0)              # p, d, q
SEASONAL_ORDER = (1, 0, 0, 7)  # P, D, Q, s
MONTH = pd.DateOffset(months=1)

date_col, value_col = df.columns
dates = pd.DatetimeIndex(pd.to_datetime(df[date_col]), name=date_col).normalize()
values = pd.to_numeric(df[value_col]).astype(float)
y = pd.Series(values.to_numpy(), index=dates, name=value_col).sort_index()

if y.notna().sum() < 2 or dates.hasnans or dates.has_duplicates:
    raise ValueError("Нужны уникальные даты и минимум два непустых значения.")
if y.abs().eq(float("inf")).any():
    raise ValueError("Значения ряда должны быть конечными числами или пропусками.")

y = y.asfreq("D")

## MAE и MAPE на истории

Даты делятся по порядку на `train` (60%), `valid` (20%) и `test` (20%). Для оценки `train` первая половина этого отрезка служит начальной историей. В начале каждого отрезка оценки коэффициенты обучаются на всех предшествующих данных. Прогноз строится на месяц, затем модель получает факты этого блока и прогнозирует следующий. Последний блок может быть короче месяца.

Метрики усредняют ошибку по всем прогнозным дням с известным фактом. MAE выражена в единицах ряда, MAPE — в процентах. Дни с нулевым фактом входят в MAE, для MAPE используются ненулевые факты.

In [ ]:
def fit(history):
    return SARIMAX(history, order=ORDER, seasonal_order=SEASONAL_ORDER).fit(disp=False)


def month_days(origin):
    return ((origin + MONTH).date() - origin.date()).days


train_end, valid_end = int(len(y) * 0.6), int(len(y) * 0.8)
warmup = train_end // 2
if y.iloc[:warmup].count() < 2:
    raise ValueError("Недостаточно истории перед первым прогнозом на train.")

splits = {"train": (warmup, train_end), "valid": (train_end, valid_end),
          "test": (valid_end, len(y))}
blocks = []
for split, (start, stop) in splits.items():
    state = fit(y.iloc[:start])
    while start < stop:
        origin = y.index[start - 1]
        end = min(start + month_days(origin), stop)
        actual = y.iloc[start:end]
        predicted = state.forecast(steps=len(actual))
        blocks.append(pd.DataFrame({"split": split, "origin": origin,
                                    "actual": actual, "forecast": predicted}))
        start = end
        if start < stop:
            state = state.extend(actual)

backtest = pd.concat(blocks)
error = (backtest["actual"] - backtest["forecast"]).abs()
denominator = backtest["actual"].abs().replace(0, float("nan"))
metrics = (backtest.assign(MAE=error, MAPE_pct=100 * error / denominator)
           .groupby("split", sort=False)[["MAE", "MAPE_pct"]].mean())
display(metrics.round(3))

## Прогноз на следующий месяц

После проверки на истории модель обучается на всём ряде. Таблица `forecast` содержит дневной прогноз и 95% интервал.

In [ ]:
model = fit(y)
display(model.params.rename("estimate").to_frame())

prediction = model.get_forecast(steps=month_days(y.index[-1]))
forecast = prediction.predicted_mean.rename("forecast").to_frame()
forecast[["lower_95", "upper_95"]] = prediction.conf_int(alpha=0.05).to_numpy()
display(forecast)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 11), layout="constrained")
for ax, (split, rows) in zip(axes, backtest.groupby("split", sort=False)):
    ax.plot(rows.index, rows["actual"], color="C0", label="Факт")
    ax.plot(rows.index, rows["forecast"], color="C1", label="Прогноз")
    ax.set_title(split, loc="left")
    ax.legend()

ax = axes[-1]
recent = y.tail(90)
ax.plot(recent.index, recent, color="C0", label="Факт")
ax.plot(forecast.index, forecast["forecast"], color="C1", label="Прогноз")
ax.fill_between(forecast.index, forecast["lower_95"], forecast["upper_95"],
                color="C1", alpha=0.18, label="95% интервал")
ax.set_title("Следующий месяц", loc="left")
ax.legend()
plt.show()

## Как собрать `(p, d, q) × (P, D, Q, s)`

| Параметр | Что задаёт |
|---|---|
| `s` | Число шагов в цикле. Для недельного цикла на дневных данных — `7`. |
| `d`, `D` | Сколько раз взять обычную разность `y[t] − y[t−1]` и сезонную `y[t] − y[t−s]` соответственно. |
| `p`, `P` | Число лагов значений ряда после разностей. Обычные лаги `1, 2, …`, сезонные `s, 2s, …`. |
| `q`, `Q` | Число лагов прошлых ошибок прогноза. Обычные лаги `1, 2, …`, сезонные `s, 2s, …`. |

1. Задай `s` по календарю и смыслу ряда.
2. Начни с `d = D = 0`. При выраженной сезонности проверь, стабилизирует ли ряд сезонная разность (`D = 1`). Если уровень ряда всё ещё дрейфует, попробуй `d = 1`. Выбери минимум разностей, при котором среднее ряда стабильно.
3. Для `p, q, P, Q` начни с вариантов `0` и `1`. На ряде после разностей PACF подсказывает кандидатов для AR, ACF — для MA. Для сезонных порядков смотри лаги `s, 2s, …`. Сравни варианты по MAE на `valid`, а итоговое качество проверь на `test`. [Forecasting: Principles and Practice](https://otexts.com/fpp3/arima.html#seasonal-arima-models)

**Числа в `model.params` оценивает модель.** Например, при `(1, 1, 0) × (1, 0, 0, 7)` `ar.L1` — вес вчерашнего изменения, `ar.S.L7` — вес изменения неделю назад. Если добавить MA, `ma.L1` и `ma.S.L7` будут весами прошлых ошибок на лагах 1 и 7. `sigma2` — дисперсия случайной ошибки. [Документация SARIMAX](https://www.statsmodels.org/stable/generated/statsmodels.tsa.statespace.sarimax.SARIMAX.html)